# Memory-Augmented Brain Encoding — Setup & First Prediction
**Author:** MD Rabbi | **Based on:** TRIBE v2 (Meta FAIR, March 2026)

This notebook:
1. Installs TRIBE v2 and all dependencies
2. Connects to Google Drive (permanent storage)
3. Loads the pretrained model
4. Runs your first brain prediction on a sample video
5. Visualizes the predicted brain activity

**Important:** Always run cells in order. If Colab disconnects, run from the top — your results are saved in Google Drive.

---
## 0. Connect Google Drive (YOUR PERMANENT STORAGE)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create project folder in your Drive
import os
PROJECT_DIR = '/content/drive/MyDrive/Research/memory-brain-encoding'
CACHE_DIR = os.path.join(PROJECT_DIR, 'cache')
RESULTS_DIR = os.path.join(PROJECT_DIR, 'results')
DATA_DIR = os.path.join(PROJECT_DIR, 'data')

for d in [PROJECT_DIR, CACHE_DIR, RESULTS_DIR, DATA_DIR]:
    os.makedirs(d, exist_ok=True)

print(f'Project directory: {PROJECT_DIR}')
print('All results will be saved to Google Drive — safe from disconnects!')

---
## 1. Install TRIBE v2

In [ ]:
%%capture install_output
# Install TRIBE v2 with all dependencies
!pip install git+https://github.com/facebookresearch/tribev2.git
!pip install nibabel matplotlib seaborn nilearn scipy pyvista scikit-image
print('Installation complete!')

In [ ]:
# Verify installation
from tribev2 import TribeModel
print('TRIBE v2 imported successfully!')

---
## 2. Authenticate HuggingFace (needed for Llama 3.2)

In [ ]:
# Option A: Use Colab secrets (recommended)
# Go to the key icon in the left sidebar, add a secret named HF_TOKEN
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    from huggingface_hub import login
    login(token=hf_token)
    print('Logged in via Colab secrets!')
except Exception:
    # Option B: Manual login
    from huggingface_hub import login
    login()
    print('Logged in manually!')

---
## 3. Check GPU

In [ ]:
import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f'GPU: {gpu_name} ({gpu_mem:.1f} GB)')
else:
    print('WARNING: No GPU detected!')
    print('Go to Runtime > Change runtime type > GPU')

---
## 4. Load Pretrained TRIBE v2 Model

In [ ]:
print('Loading TRIBE v2 from HuggingFace...')
print('(First time downloads ~700MB, cached in Drive after that)')

model = TribeModel.from_pretrained(
    'facebook/tribev2',
    cache_folder=CACHE_DIR
)

brain_model = model._model
total_params = sum(p.numel() for p in brain_model.parameters())
print(f'Model loaded! {total_params:,} parameters')
print(f'Output: {brain_model.n_outputs} cortical vertices')

---
## 5. Download a Sample Video

In [ ]:
import urllib.request

video_path = os.path.join(DATA_DIR, 'sample_video.mp4')

if not os.path.exists(video_path):
    print('Downloading sample video...')
    urllib.request.urlretrieve(
        'https://test-videos.co.uk/vids/bigbuckbunny/mp4/h264/360/Big_Buck_Bunny_360_10s_1MB.mp4',
        video_path
    )
    print(f'Saved to {video_path}')
else:
    print(f'Video already exists at {video_path}')

---
## 6. Extract Features & Predict Brain Activity
This is the big moment — TRIBE v2 will:
1. Run V-JEPA2 on the video frames
2. Run Wav2Vec-BERT on the audio
3. Run Llama 3.2 on any detected speech
4. Feed all three through the Transformer encoder
5. Predict activity at 20,484 cortical vertices

In [ ]:
import numpy as np

print('Step 1/3: Extracting multimodal features from video...')
print('(Running V-JEPA2, Wav2Vec-BERT, and Llama 3.2)')
df = model.get_events_dataframe(video_path=video_path)
print(f'Extracted {len(df)} events from video')
print(f'Event types: {df["type"].value_counts().to_dict()}')

print('\nStep 2/3: Running TRIBE v2 Transformer encoder...')
preds, segments = model.predict(events=df)

print(f'\nStep 3/3: Done!')
print(f'Predictions shape: {preds.shape}')
print(f'  → {preds.shape[0]} timepoints (1 per second)')
print(f'  → {preds.shape[1]} cortical vertices')

# Save to Google Drive
save_path = os.path.join(RESULTS_DIR, 'first_brain_prediction.npy')
np.save(save_path, preds)
print(f'\nSaved predictions to Google Drive: {save_path}')

---
## 7. Visualize the Brain Prediction!
Let's see what the model thinks the brain does while watching this video.

In [ ]:
import matplotlib.pyplot as plt
from nilearn import plotting, datasets, surface

# Get the fsaverage5 surface mesh
fsaverage = datasets.fetch_surf_fsaverage(mesh='fsaverage5')

# Pick the timepoint with the strongest brain response
strongest_t = np.abs(preds).mean(axis=1).argmax()
brain_map = preds[strongest_t, :]

# Split into left and right hemispheres
n_vertices_per_hemi = brain_map.shape[0] // 2
left_data = brain_map[:n_vertices_per_hemi]
right_data = brain_map[n_vertices_per_hemi:]

print(f'Showing brain prediction at t={strongest_t}s (strongest response)')
print(f'Left hemisphere: {left_data.shape[0]} vertices')
print(f'Right hemisphere: {right_data.shape[0]} vertices')

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5),
                         subplot_kw={'projection': '3d'})

# Left hemisphere - lateral view
plotting.plot_surf_stat_map(
    fsaverage['pial_left'], left_data,
    hemi='left', view='lateral',
    colorbar=True, cmap='cold_hot',
    title=f'Left hemisphere (t={strongest_t}s)',
    axes=axes[0], figure=fig
)

# Right hemisphere - lateral view
plotting.plot_surf_stat_map(
    fsaverage['pial_right'], right_data,
    hemi='right', view='lateral',
    colorbar=True, cmap='cold_hot',
    title=f'Right hemisphere (t={strongest_t}s)',
    axes=axes[1], figure=fig
)

plt.tight_layout()

# Save figure to Drive
fig_path = os.path.join(RESULTS_DIR, 'first_brain_visualization.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f'Saved visualization to {fig_path}')
plt.show()

---
## 8. Explore the Architecture (Understanding the Code)

In [ ]:
print('='*60)
print('TRIBE v2 ARCHITECTURE BREAKDOWN')
print('='*60)

print('\n--- Modality Projectors ---')
for name, proj in brain_model.projectors.items():
    params = sum(p.numel() for p in proj.parameters())
    print(f'  {name}: {params:,} parameters → 384 dimensions')

print(f'\n--- Transformer Encoder ---')
enc_params = sum(p.numel() for p in brain_model.encoder.parameters())
print(f'  Parameters: {enc_params:,}')
print(f'  Hidden dim: {brain_model.config.hidden}')
print(f'  Context window: {brain_model.config.max_seq_len} timesteps')

print(f'\n--- Low-Rank Head ---')
print(f'  Shape: {brain_model.low_rank_head.weight.shape}')

print(f'\n--- Subject Block ---')
pred_params = sum(p.numel() for p in brain_model.predictor.parameters())
print(f'  Parameters: {pred_params:,}')
print(f'  Output: {brain_model.n_outputs} vertices')

total = sum(p.numel() for p in brain_model.parameters())
print(f'\n--- TOTAL: {total:,} parameters ---')

print('\n' + '='*60)
print('MEMORY MODULE INSERTION POINT')
print('='*60)
print('''
forward() pipeline:

  1. aggregate_features()  → [B, T, 1152]  (video+audio+text)
  2. transformer_forward() → [B, T, 1152]  (temporal mixing)
  
  >>> YOUR MEMORY MODULE GOES HERE <<<
  >>> Cross-attend to past window latents <<<
  
  3. low_rank_head()       → [B, T, 2048]  (compress)
  4. predictor()           → [B, 20484, T]  (map to brain)
  5. pooler()              → [B, 20484, T'] (2Hz → 1Hz)
''')

---
## 9. Analyze Per-Region Encoding (Baseline for Your Research)
This shows which brain regions are predicted well vs poorly.
Regions with LOW scores (hippocampus, prefrontal) = where your memory module should help.

In [ ]:
# Show prediction strength by brain region
pred_strength = np.abs(preds).mean(axis=0)  # average across time

# Left and right hemisphere
left_strength = pred_strength[:n_vertices_per_hemi]
right_strength = pred_strength[n_vertices_per_hemi:]

fig, axes = plt.subplots(1, 2, figsize=(14, 5),
                         subplot_kw={'projection': '3d'})

plotting.plot_surf_stat_map(
    fsaverage['pial_left'], left_strength,
    hemi='left', view='lateral',
    colorbar=True, cmap='YlOrRd',
    title='Prediction strength (left)',
    axes=axes[0], figure=fig
)

plotting.plot_surf_stat_map(
    fsaverage['pial_right'], right_strength,
    hemi='right', view='lateral',
    colorbar=True, cmap='YlOrRd',
    title='Prediction strength (right)',
    axes=axes[1], figure=fig
)

plt.tight_layout()
fig_path = os.path.join(RESULTS_DIR, 'prediction_strength_by_region.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f'Saved to {fig_path}')
plt.show()

print('\nBright areas = strong predictions (visual/auditory cortex)')
print('Dark areas = weak predictions (hippocampus, prefrontal)')
print('Your memory module aims to brighten those dark areas!')

---
## 10. Save Session Summary

In [ ]:
import json
from datetime import datetime

session = {
    'date': datetime.now().isoformat(),
    'notebook': '01_setup_and_first_prediction',
    'model': 'facebook/tribev2',
    'total_params': int(total),
    'prediction_shape': list(preds.shape),
    'video': video_path,
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
    'status': 'completed'
}

log_path = os.path.join(RESULTS_DIR, 'session_log.json')
# Append to existing log
if os.path.exists(log_path):
    with open(log_path) as f:
        logs = json.load(f)
else:
    logs = []
logs.append(session)
with open(log_path, 'w') as f:
    json.dump(logs, f, indent=2)

print('Session saved!')
print(f'\nFiles in your Google Drive ({RESULTS_DIR}):')
for f in os.listdir(RESULTS_DIR):
    print(f'  {f}')

---
## What's Next?

You just:
- Loaded a 177M parameter brain encoding model
- Predicted activity at 20,484 cortical vertices from a video
- Visualized the predicted brain response
- Saved everything to Google Drive

**Next notebook (02):** Reproduce the baseline encoding scores on the Algonauts dataset

**Your memory module will insert between steps 2 and 3 of the forward() pipeline.**